# Bus Assignment with AMPL in Python
## Cyclic Shift-Covering Model with Multiple Optima

This notebook solves the same bus assignment problem with AMPL from Python using `amplpy`.

We will:
- solve the base minimization model
- recover an alternative optimal solution with the same total number of buses
- verify the textbook solution from the notes

Each run reports:
- one feasible optimal assignment
- the total number of buses
- the execution time


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=['highs'], license_uuid='default')` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(variable, shift_order):
    values = variable.get_values().to_dict()
    solution = {}

    for shift in shift_order:
        if shift in values:
            solution[shift] = int(round(values[shift]))
        else:
            solution[shift] = int(round(values[(shift,)]))

    solution['total_buses'] = sum(solution[shift] for shift in shift_order)
    return solution


def check_solution(solution, shift_order, interval_order, demand_data, covers):
    feasible = True

    for interval in interval_order:
        covered = sum(covers[(shift, interval)] * solution[shift] for shift in shift_order)
        if covered < demand_data[interval]:
            feasible = False
            break

    return {
        'solution': solution,
        'feasible': feasible,
        'total_buses': sum(solution[shift] for shift in shift_order),
    }


## Problem: Bus Assignment

We keep the same six 8-hour shifts from the notes and the same demand per 4-hour interval.

In AMPL, the model is written as a small integer covering problem:
- one variable per feasible 8-hour shift
- one coverage constraint per 4-hour interval
- an objective that minimizes the total number of assigned buses


In [4]:
SHIFT_DATA = {
    'x1': '12 a.m. - 8 a.m.',
    'x2': '4 a.m. - 12 p.m.',
    'x3': '8 a.m. - 4 p.m.',
    'x4': '12 p.m. - 8 p.m.',
    'x5': '4 p.m. - 12 a.m.',
    'x6': '8 p.m. - 4 a.m.',
}

INTERVAL_DATA = {
    'i1': {'label': '12 a.m. - 4 a.m.', 'demand': 14},
    'i2': {'label': '4 a.m. - 8 a.m.', 'demand': 6},
    'i3': {'label': '8 a.m. - 12 p.m.', 'demand': 12},
    'i4': {'label': '12 p.m. - 4 p.m.', 'demand': 5},
    'i5': {'label': '4 p.m. - 8 p.m.', 'demand': 11},
    'i6': {'label': '8 p.m. - 12 a.m.', 'demand': 8},
}

SHIFT_COVERAGE = {
    'x1': ['i1', 'i2'],
    'x2': ['i2', 'i3'],
    'x3': ['i3', 'i4'],
    'x4': ['i4', 'i5'],
    'x5': ['i5', 'i6'],
    'x6': ['i6', 'i1'],
}

DEMAND_DATA = {
    interval: data['demand']
    for interval, data in INTERVAL_DATA.items()
}

COVERS = {
    (shift, interval): int(interval in SHIFT_COVERAGE[shift])
    for shift in SHIFT_DATA
    for interval in INTERVAL_DATA
}

TEXTBOOK_SOLUTION = {
    'x1': 14,
    'x2': 9,
    'x3': 3,
    'x4': 3,
    'x5': 8,
    'x6': 0,
}


In [5]:
@timed
def solve_bus_assignment_with_ampl(shift_data, interval_data, demand_data, covers, solver: str = 'highs'):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        set SHIFTS ordered;
        set INTERVALS ordered;

        param demand {INTERVALS} >= 0;
        param covers {SHIFTS, INTERVALS} binary;

        var Assign {s in SHIFTS} integer >= 0;

        minimize TotalBuses:
            sum {s in SHIFTS} Assign[s];

        subject to CoverDemand {i in INTERVALS}:
            sum {s in SHIFTS} covers[s, i] * Assign[s] >= demand[i];
        '''
    )

    ampl.set['SHIFTS'] = list(shift_data)
    ampl.set['INTERVALS'] = list(interval_data)
    ampl.param['demand'] = demand_data
    ampl.param['covers'] = covers

    ampl.solve(solver=solver)

    solution = extract_solution(ampl.get_variable('Assign'), list(shift_data))
    solution['total_buses'] = int(round(ampl.get_objective('TotalBuses').value()))
    return solution


In [6]:
ampl_result, ampl_time = solve_bus_assignment_with_ampl(
    shift_data=SHIFT_DATA,
    interval_data=INTERVAL_DATA,
    demand_data=DEMAND_DATA,
    covers=COVERS,
)

print('=== BUS ASSIGNMENT RESULTS WITH AMPL ===')
print(f'Solution -> {ampl_result}')
print(f'Time     -> {ampl_time:.8f} seconds')


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 37
4 simplex iterations
1 branching nodes
=== BUS ASSIGNMENT RESULTS WITH AMPL ===
Solution -> {'x1': 6, 'x2': 6, 'x3': 6, 'x4': 5, 'x5': 6, 'x6': 8, 'total_buses': 37}
Time     -> 1.86307217 seconds


## Alternative Optimal Solution

To illustrate multiple optima in AMPL, we fix the minimum total number of buses found in the first run and then optimize a secondary criterion.

Here we maximize `x6`, which forces AMPL to return a different optimal assignment while preserving the same total.


In [7]:
@timed
def solve_alternative_optimum_with_ampl(
    shift_data,
    interval_data,
    demand_data,
    covers,
    fixed_total,
    preferred_shift: str = 'x6',
    solver: str = 'highs',
):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        f'''
        set SHIFTS ordered;
        set INTERVALS ordered;

        param demand {{INTERVALS}} >= 0;
        param covers {{SHIFTS, INTERVALS}} binary;
        param fixed_total >= 0;

        var Assign {{s in SHIFTS}} integer >= 0;

        maximize PreferredShiftLoad:
            Assign["{preferred_shift}"];

        subject to FixedTotal:
            sum {{s in SHIFTS}} Assign[s] = fixed_total;

        subject to CoverDemand {{i in INTERVALS}}:
            sum {{s in SHIFTS}} covers[s, i] * Assign[s] >= demand[i];
        '''
    )

    ampl.set['SHIFTS'] = list(shift_data)
    ampl.set['INTERVALS'] = list(interval_data)
    ampl.param['demand'] = demand_data
    ampl.param['covers'] = covers
    ampl.param['fixed_total'] = fixed_total

    ampl.solve(solver=solver)

    solution = extract_solution(ampl.get_variable('Assign'), list(shift_data))
    solution['total_buses'] = fixed_total
    return solution


In [8]:
alternative_result, alternative_time = solve_alternative_optimum_with_ampl(
    shift_data=SHIFT_DATA,
    interval_data=INTERVAL_DATA,
    demand_data=DEMAND_DATA,
    covers=COVERS,
    fixed_total=ampl_result['total_buses'],
    preferred_shift='x6',
)

textbook_check = check_solution(
    solution=TEXTBOOK_SOLUTION,
    shift_order=list(SHIFT_DATA),
    interval_order=list(INTERVAL_DATA),
    demand_data=DEMAND_DATA,
    covers=COVERS,
)

print('=== ALTERNATIVE OPTIMAL SOLUTION ===')
print(f'Solution -> {alternative_result}')
print(f'Time     -> {alternative_time:.8f} seconds')
print()
print('=== TEXTBOOK SOLUTION CHECK ===')
print(textbook_check)


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 14
4 simplex iterations
1 branching nodes
=== ALTERNATIVE OPTIMAL SOLUTION ===
Solution -> {'x1': 0, 'x2': 12, 'x3': 0, 'x4': 11, 'x5': 0, 'x6': 14, 'total_buses': 37}
Time     -> 1.75037421 seconds

=== TEXTBOOK SOLUTION CHECK ===
{'solution': {'x1': 14, 'x2': 9, 'x3': 3, 'x4': 3, 'x5': 8, 'x6': 0}, 'feasible': True, 'total_buses': 37}


In [9]:
print('=== FINAL COMPARISON ===')
print()
print('Minimum total buses')
print(f'AMPL optimum    -> {ampl_result}, time = {ampl_time:.8f} seconds')
print()
print('Alternative optimum with the same total')
print(f'Prefer x6       -> {alternative_result}, time = {alternative_time:.8f} seconds')
print()
print('Textbook solution from the notes')
print(f'Reference plan  -> {textbook_check}')


=== FINAL COMPARISON ===

Minimum total buses
AMPL optimum    -> {'x1': 6, 'x2': 6, 'x3': 6, 'x4': 5, 'x5': 6, 'x6': 8, 'total_buses': 37}, time = 1.86307217 seconds

Alternative optimum with the same total
Prefer x6       -> {'x1': 0, 'x2': 12, 'x3': 0, 'x4': 11, 'x5': 0, 'x6': 14, 'total_buses': 37}, time = 1.75037421 seconds

Textbook solution from the notes
Reference plan  -> {'solution': {'x1': 14, 'x2': 9, 'x3': 3, 'x4': 3, 'x5': 8, 'x6': 0}, 'feasible': True, 'total_buses': 37}
